In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

# --- 1. Define Helper Function for RR@10 ---
def calculate_rr_at_10(rank):
    """Calculates the Reciprocal Rank at 10."""
    rank = pd.to_numeric(rank, errors="coerce")
    if pd.isna(rank) or rank == 0:
        return 0.0
    return (1.0 / rank) if (1.0 <= rank <= 10.0) else 0.0

print("--- Error Mode Analysis Script Started ---")

# --- 2. Define All File Paths ---

# Base directory (USER CONFIRMED CORRECT)
base_dir = "/content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/"

# Path to the master error bank file
master_file = "/content/drive/MyDrive/ComplianceGPT_v2/data/error_bank/error_bank_v1.csv"


system_files = {
    "S1_MRR": [
        base_dir + "system_1_bm25/S1_bm25_error_bank_rev4_results.csv",
        base_dir + "system_1_bm25/S1_bm25_error_bank_rev5_results.csv",
    ],
    "S2_MRR": [
        base_dir + "system_2_dense/S2_dense_error_bank_rev4_results.csv",
        base_dir + "system_2_dense/S2_dense_error_bank_rev5_results.csv",
    ],
    "S5_MRR": [
        base_dir + "system_5_hybrid_rrf/S5_hybrid_rrf_error_bank_rev4_results.csv",
        base_dir + "system_5_hybrid_rrf/S5_hybrid_rrf_error_bank_rev5_results.csv",
    ],
    "S6_MRR": [
        base_dir + "system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev4_results.csv",
        base_dir + "system_6_hybrid_rerank/S6_hybrid_rerank_error_bank_rev5_results.csv",
    ],
    "S7_MRR": [
        base_dir + "system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev4_results.csv",
        base_dir + "system_7_compliance_gpt/S7_compliance_gpt_error_bank_rev5_results.csv",
    ],
}

print(f"Base dir exists? {os.path.exists(base_dir)} -> {base_dir}")
print(f"Master file exists? {os.path.exists(master_file)} -> {master_file}\n")

# --- 3. Load Master File ---
try:
    master_df = pd.read_csv(master_file)

    required_master_cols = ["question_id", "question", "failure_category"]
    missing_master = [c for c in required_master_cols if c not in master_df.columns]
    if missing_master:
        raise ValueError(f"Master file missing columns: {missing_master}. Available: {list(master_df.columns)}")

    master_df = master_df[required_master_cols].copy()

    # Normalize + validate failure_category (prevents empty pivot_table)
    master_df["failure_category"] = master_df["failure_category"].astype("string")
    master_df["failure_category"] = master_df["failure_category"].str.strip()
    master_df.loc[master_df["failure_category"].isna() | (master_df["failure_category"] == ""), "failure_category"] = "UNLABELED"

    if (master_df["failure_category"] == "UNLABELED").all():
        print("CRITICAL WARNING: 'failure_category' is entirely missing/blank in the master error bank.")
        print("Pivot table will group everything under 'UNLABELED'. Fix the master file to restore real failure-mode buckets.")

    # Normalize join key type to avoid silent merge mismatches
    master_df["question_id"] = master_df["question_id"].astype(str)

    print(f"Loaded master file: '{master_file}' (N={len(master_df)})")
    print("Failure categories:")
    print(master_df["failure_category"].value_counts().to_string())
    print()

    merged_data_df = master_df.copy()

except Exception as e:
    print(f"CRITICAL ERROR: Could not load master file '{master_file}'.")
    print(f"Error: {e}")
    raise

# --- 4. Process Each System's Results ---
print("--- Starting System MRR Calculation Loop ---")

for system_name, files in system_files.items():
    # File existence diagnostics (helps explain 'empty output' cases)
    existence = [(f, os.path.exists(f)) for f in files]
    existing_files = [f for (f, ok) in existence if ok]

    if not existing_files:
        print(f"[SKIP] {system_name}: no result files found.")
        for f, ok in existence:
            print(f"   - EXISTS={ok}  {f}")
        continue

    try:
        df_list = [pd.read_csv(f) for f in existing_files]
        system_df = pd.concat(df_list, ignore_index=True)

        required_cols = ["question_id", "rank"]
        missing = [c for c in required_cols if c not in system_df.columns]
        if missing:
            print(f"[SKIP] {system_name}: missing columns {missing}. Available: {list(system_df.columns)}")
            continue

        # Normalize join key type to avoid silent merge mismatches
        system_df["question_id"] = system_df["question_id"].astype(str)

        system_df[system_name] = system_df["rank"].apply(calculate_rr_at_10)

        # Use max RR across multiple rows for the same question_id
        system_rr_df = system_df.groupby("question_id")[[system_name]].max().reset_index()

        merged_data_df = merged_data_df.merge(system_rr_df, on="question_id", how="left")

        print(f"[OK] {system_name}: loaded {len(existing_files)} file(s), rows={len(system_df)}, unique_qids={system_rr_df.shape[0]}")

    except Exception as e:
        print(f"[ERROR] {system_name}: {e}")

print("--- System MRR Calculation Loop Finished ---\n")

# --- 5. Create Pivot Table and Save Artifacts ---
mrr_columns = [col for col in merged_data_df.columns if col.endswith("_MRR") and col.startswith("S")]

if not mrr_columns:
    print("[ERROR] No MRR columns were added. This usually means:")
    print("  - result files were not found (paths wrong or Drive not mounted), OR")
    print("  - CSVs are missing required columns: question_id, rank.")
    raise RuntimeError("No MRR data was successfully processed.")

merged_data_df[mrr_columns] = merged_data_df[mrr_columns].fillna(0.0)

# Order columns for readability (S1, S2, S5, S6, S7)
ordered_cols = sorted(mrr_columns)

# --- 5. Re-attach / validate failure_category labels BEFORE pivot (root cause of empty pivot_table) ---
# Even if system merges work, pivot_table will be empty if failure_category is all-NaN.
label_df = master_df[["question_id", "question", "failure_category"]].drop_duplicates(
    subset=["question_id", "question"], keep="first"
)

# Ensure merged_data_df has exactly one failure_category column sourced from the master labels
merged_data_df = merged_data_df.drop(columns=["failure_category"], errors="ignore").merge(
    label_df, on=["question_id", "question"], how="left"
)

# Final cleanup: normalize blanks -> UNLABELED
merged_data_df["failure_category"] = merged_data_df["failure_category"].astype("string").str.strip()
merged_data_df.loc[
    merged_data_df["failure_category"].isna() | (merged_data_df["failure_category"] == ""),
    "failure_category",
] = "UNLABELED"

null_rate = float((merged_data_df["failure_category"] == "UNLABELED").mean())
print(f"failure_category UNLABELED rate: {null_rate:.3f}")
print("failure_category value_counts (top):")
print(merged_data_df["failure_category"].value_counts(dropna=False).head(10).to_string())
print()
pivot_table = merged_data_df.groupby("failure_category", dropna=False)[ordered_cols].mean()
print(f"pivot_table rows={pivot_table.shape[0]}")

if pivot_table.shape[0] == 0:
    print("CRITICAL ERROR: pivot_table has 0 rows. This happens when failure_category is missing for all rows.")
    print("Check the master error bank file at:")
    print(f"  {master_file}")
    print("And verify failure_category is populated.")
    raise RuntimeError("pivot_table is empty (failure_category missing).")


print("--- Error Bank by Failure Mode (MRR@10) ---")
print(pivot_table.to_markdown(floatfmt=".4f"))

# --- Save artifacts ---
output_dir = "/content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/error_analysis/generated_reports"
os.makedirs(output_dir, exist_ok=True)

csv_pivot_path = os.path.join(output_dir, "pivot_table.csv")
md_pivot_path = os.path.join(output_dir, "pivot_table.md")
merged_data_path = os.path.join(output_dir, "merged_results_data.csv")

pivot_table.to_csv(csv_pivot_path, float_format="%.4f")
print(f"\n[OK] Saved pivot table CSV: {csv_pivot_path}")

md_text = pivot_table.to_markdown(floatfmt=".4f")
with open(md_pivot_path, "w", encoding="utf-8") as f:
    f.write(md_text)
print(f"[OK] Saved pivot table Markdown: {md_pivot_path}")

merged_data_df.to_csv(merged_data_path, index=False, float_format="%.4f")
print(f"[OK] Saved merged data CSV: {merged_data_path}")

print("\n--- Error Mode Analysis Script Finished ---")


--- Error Mode Analysis Script Started ---
Base dir exists? True -> /content/drive/MyDrive/ComplianceGPT_v2/experiments/retriever_ablation/ablation_outputs/
Master file exists? True -> /content/drive/MyDrive/ComplianceGPT_v2/data/error_bank/error_bank_v1.csv

CRITICAL WARNING: 'failure_category' is entirely missing/blank in the master error bank.
Pivot table will group everything under 'UNLABELED'. Fix the master file to restore real failure-mode buckets.
Loaded master file: '/content/drive/MyDrive/ComplianceGPT_v2/data/error_bank/error_bank_v1.csv' (N=37)
Failure categories:
failure_category
UNLABELED    37

--- Starting System MRR Calculation Loop ---
[OK] S1_MRR: loaded 2 file(s), rows=37, unique_qids=34
[OK] S2_MRR: loaded 2 file(s), rows=37, unique_qids=34
[OK] S5_MRR: loaded 2 file(s), rows=37, unique_qids=34
[OK] S6_MRR: loaded 2 file(s), rows=37, unique_qids=34
[OK] S7_MRR: loaded 2 file(s), rows=37, unique_qids=34
--- System MRR Calculation Loop Finished ---

failure_category 